# PJM OASIS — from scratch

Run cells top to bottom. Fill in credentials in cell 2.

```bash
pip install cryptography
```

In [ ]:
import json
import ssl
import tempfile
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

from cryptography import x509
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.serialization import (
    Encoding,
    NoEncryption,
    PrivateFormat,
    pkcs12,
)

In [ ]:
# --- fill in ---
USERNAME = ""
PASSWORD = ""
CERT_PATH = ""       # .p12 or .pfx login file
CERT_PASSWORD = ""   # blank if none
ENV = "TRAIN"        # TRAIN | STAGE | TEST | PRODUCTION
# ---------------

In [ ]:
URLS = {
    "TRAIN": {
        "oasis": "https://oasis.ac1train.pjm.com/OASIS/",
        "sso": "https://sotrain.pjm.com/access/authenticate/pjmauthcert",
        "logout": "https://sotrain.pjm.com/access/logout",
    },
    "STAGE": {
        "oasis": "https://oasis.ac1stage.pjm.com/OASIS/",
        "sso": "https://sotrain.pjm.com/access/authenticate/pjmauthcert",
        "logout": "https://sotrain.pjm.com/access/logout",
    },
    "TEST": {
        "oasis": "https://oasis.test.pjm.com/OASIS/",
        "sso": "https://sotrain.pjm.com/access/authenticate/pjmauthcert",
        "logout": "https://sotrain.pjm.com/access/logout",
    },
    "PRODUCTION": {
        "oasis": "https://pjmoasis.pjm.com/OASIS/",
        "sso": "https://sso.pjm.com/access/authenticate/pjmauthcert",
        "logout": "https://sso.pjm.com/access/logout",
    },
}

env = ENV.upper()
if env not in URLS:
    raise ValueError(f"ENV must be one of: {', '.join(URLS)}")

oasis_base = URLS[env]["oasis"]
sso_url = URLS[env]["sso"]
logout_url = URLS[env]["logout"]
cert_path = Path(CERT_PATH).expanduser()
timeout = 30

print("user:", USERNAME)
print("env:", env)
print("cert:", cert_path)
print("sso:", sso_url)
print("oasis:", oasis_base)

if not USERNAME or not PASSWORD or not CERT_PATH:
    raise ValueError("Set USERNAME, PASSWORD, and CERT_PATH")
if not cert_path.exists():
    raise FileNotFoundError(f"Certificate not found: {cert_path}")

In [ ]:
p12_data = cert_path.read_bytes()
p12_pwd = CERT_PASSWORD.encode() if CERT_PASSWORD else None
private_key, certificate, _ = pkcs12.load_key_and_certificates(p12_data, p12_pwd)
if private_key is None or certificate is None:
    raise ValueError("PKCS#12 file must contain both private key and certificate")

cert_pem = certificate.public_bytes(Encoding.PEM)
key_pem = private_key.private_bytes(Encoding.PEM, PrivateFormat.PKCS8, NoEncryption())
x509_cert = x509.load_pem_x509_certificate(cert_pem, default_backend())

print("subject:", x509_cert.subject.rfc4514_string())
print("issuer:", x509_cert.issuer.rfc4514_string())
print("thumbprint:", x509_cert.fingerprint(hashes.SHA256()).hex())
print("not_after:", x509_cert.not_valid_after_utc)

In [ ]:
cert_file = tempfile.NamedTemporaryFile(mode="wb", suffix=".pem", delete=False)
key_file = tempfile.NamedTemporaryFile(mode="wb", suffix=".pem", delete=False)
cert_file.write(cert_pem)
key_file.write(key_pem)
cert_file.close()
key_file.close()
cert_pem_path = Path(cert_file.name)
key_pem_path = Path(key_file.name)
cert_pem_path.chmod(0o600)
key_pem_path.chmod(0o600)

ssl_ctx = ssl.create_default_context()
ssl_ctx.load_cert_chain(certfile=str(cert_pem_path), keyfile=str(key_pem_path))
print("ssl context ready")


def http_request(method, url, *, headers=None, body=None):
    req = urllib.request.Request(url, data=body, method=method.upper())
    for key, value in (headers or {}).items():
        req.add_header(key, value)
    try:
        with urllib.request.urlopen(req, context=ssl_ctx, timeout=timeout) as resp:
            content = resp.read()
            hdrs = {k.lower(): v for k, v in resp.headers.items()}
            return resp.status, hdrs, content
    except urllib.error.HTTPError as exc:
        content = exc.read()
        hdrs = {k.lower(): v for k, v in exc.headers.items()} if exc.headers else {}
        return exc.code, hdrs, content

In [ ]:
status, headers, body = http_request(
    "POST",
    sso_url,
    headers={
        "Accept": "application/json",
        "X-OpenAM-Username": USERNAME,
        "X-OpenAM-Password": PASSWORD,
    },
)
print("status:", status)
print("headers:", headers)
print("body:", body.decode(errors="replace")[:500])
if status >= 400:
    raise RuntimeError(f"SSO failed ({status})")

payload = json.loads(body.decode())
token = payload.get("tokenId")
if not token:
    raise RuntimeError("No tokenId — check username, password, cert approval, env")
print("token length:", len(token))
cookie = f"pjmauth={token}"

In [ ]:
params = {
    "TEMPLATE": "TRANSSERV",
    "OUTPUT_FORMAT": "DATA",
    "PRIMARY_PROVIDER_CODE": "PJM",
    "PRIMARY_PROVIDER_DUNS": "073647877",
    "RETURN_TZ": "EP",
    "VERSION": "3.3",
    "CONTINUATION_FLAG": "N",
}
transserv_url = urllib.parse.urljoin(oasis_base, "rest/secure/transserv")
transserv_url = f"{transserv_url}?{urllib.parse.urlencode(params)}"
print("url:", transserv_url)

status, headers, body = http_request(
    "GET",
    transserv_url,
    headers={"Cookie": cookie},
)
print("status:", status)
print("headers:", headers)
print("body bytes:", len(body))
print("\nbody preview:\n", body.decode(errors="replace")[:2000])
if status >= 400:
    raise RuntimeError(f"TRANSSERV failed ({status})")

In [ ]:
status, headers, body = http_request(
    "POST",
    logout_url,
    headers={"Cookie": cookie},
)
print("logout status:", status)

cert_pem_path.unlink(missing_ok=True)
key_pem_path.unlink(missing_ok=True)
print("Done.")